# LunaShift Mission Planner v0.5.0
## Choose the Sun and Earth data source
**Automatic — recommended** | **JPL Horizons — online** | **NASA SPICE — local/offline**

This notebook runs the application on Google's computer, using your Mac or Windows browser. No GPU is needed.
The opening example is invented teaching data. Real sky data still need matching real terrain for a real-site analysis.


## 1. Upload and launch
Run the next cell and upload **LunaShift_v5.zip**. Do not upload an extracted folder.

Leave `INSTALL_SPICE`, `DOWNLOAD_SPICE_KERNELS`, and `INSTALL_REAL_DATA` set to `True` for the full setup. The first run needs internet to install packages and download about 19 MB of NASA kernels. Leave the notebook connected.

SPICE then calculates inside this Colab runtime. It is not installed on your Mac. Map services and new terrain downloads may still need internet. A deleted runtime needs setup again.

For a teaching-only session you may set all three flags to `False`. You can still use Horizons when the runtime can access NASA. Export your workspace before replacing a package or disconnecting.


In [ ]:
from google.colab import files, output
from pathlib import Path
import json, socket, stat, subprocess, sys, time, urllib.request, zipfile

# True installs optional GeoTIFF tools needed for the actual NASA terrain loader.
# Set False to use only the core teaching workspace.
INSTALL_REAL_DATA = True
INSTALL_SPICE = True
DOWNLOAD_SPICE_KERNELS = True

# Upload a ZIP. The Mac's /Users/... directory is not available inside Colab.
uploaded = files.upload()
archives = [Path(name) for name in uploaded if name.lower().endswith('.zip')]
if len(archives) != 1:
    raise RuntimeError('Upload exactly one file: LunaShift_v5.zip.')

# Stop only the process started by this notebook, if it is still running.
if 'LUNASHIFT_PROCESS' in globals() and LUNASHIFT_PROCESS.poll() is None:
    LUNASHIFT_PROCESS.terminate()
    try:
        LUNASHIFT_PROCESS.wait(timeout=5)
    except subprocess.TimeoutExpired:
        LUNASHIFT_PROCESS.kill()
        LUNASHIFT_PROCESS.wait(timeout=5)
if 'LUNASHIFT_LOG' in globals():
    LUNASHIFT_LOG.close()

WORKSPACE = Path('/content/lunashift_v5_workspace')
WORKSPACE.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archives[0]) as archive:
    if sum(item.file_size for item in archive.infolist()) > 200_000_000:
        raise RuntimeError('This is too large for the application ZIP. Do not include raw terrain data.')
    for item in archive.infolist():
        destination = (WORKSPACE / item.filename).resolve()
        if not destination.is_relative_to(WORKSPACE.resolve()):
            raise RuntimeError('Unsafe archive path rejected.')
        if stat.S_ISLNK(item.external_attr >> 16):
            raise RuntimeError('Archive symlinks are not accepted.')
        if '.workspace' in Path(item.filename).parts:
            raise RuntimeError('Use the clean application ZIP; import personal workspace JSON separately.')
    archive.extractall(WORKSPACE)

roots = [p.parent for p in WORKSPACE.rglob('run.py')
         if (p.parent / 'lunashift').is_dir() and (p.parent / 'web').is_dir()]
if len(roots) != 1:
    raise RuntimeError('Could not identify exactly one application folder. Upload the provided LunaShift_v5.zip.')
PROJECT_ROOT = roots[0]
if not (PROJECT_ROOT / 'RELEASE.json').exists():
    raise RuntimeError('This package is not the v0.5.0 release. Upload LunaShift_v5.zip.')
release = json.loads((PROJECT_ROOT / 'RELEASE.json').read_text())
if release.get('version') != '0.5.0':
    raise RuntimeError('This notebook expects LunaShift 0.5.0.')

if INSTALL_SPICE:
    print("Installing SPICE Python support in this Colab runtime...")
    subprocess.run([sys.executable, "-m", "pip", "install", "--only-binary=:all:",
                    "-r", str(PROJECT_ROOT / "requirements-spice.txt")], check=True)
    if DOWNLOAD_SPICE_KERNELS:
        result = subprocess.run([sys.executable, str(PROJECT_ROOT / "tools/setup_spice.py")],
                                cwd=PROJECT_ROOT, text=True)
        if result.returncode:
            print("SPICE files are not ready. The app will still open. Retry in Data & Sources.")

if INSTALL_REAL_DATA:
    print('Installing the optional raster packages for real terrain…')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--only-binary=:all:',
                    '-r', str(PROJECT_ROOT / 'requirements-terrain.txt')], check=True)

# Pick an unused local port instead of attaching to an older platform instance.
with socket.socket() as sock:
    sock.bind(('127.0.0.1', 0))
    PORT = sock.getsockname()[1]
LOG_PATH = Path('/content/lunashift_v5_server.log')
LUNASHIFT_LOG = LOG_PATH.open('w', buffering=1)
LUNASHIFT_PROCESS = subprocess.Popen(
    [sys.executable, 'run.py', '--colab', '--no-browser', '--port', str(PORT)],
    cwd=PROJECT_ROOT, stdout=LUNASHIFT_LOG, stderr=subprocess.STDOUT,
)
BASE = f'http://127.0.0.1:{PORT}'
for attempt in range(80):
    if LUNASHIFT_PROCESS.poll() is not None:
        raise RuntimeError('Startup failed:\n' + LOG_PATH.read_text()[-6000:])
    try:
        with urllib.request.urlopen(BASE + '/api/health', timeout=1) as response:
            health = json.load(response)
        if health.get('version') == '0.5.0':
            break
    except (OSError, ValueError):
        pass
    time.sleep(0.25)
else:
    raise RuntimeError('The server did not respond. Run the diagnostics cell below.')
print('LunaShift 0.5.0 is running. The platform opens below.')
print('Runtime folder:', PROJECT_ROOT)
print('Reminder: download a workspace copy before disconnecting.')
output.serve_kernel_port_as_iframe(PORT, height='1100')

## 2. Inspect the active input data

Run after importing NASA data. Real imagery on the map does not automatically mean real terrain is attached. This cell reads the workspace and labels sources and synthetic flags; it does not change your data.


In [ ]:
with urllib.request.urlopen(BASE + '/api/export', timeout=20) as response:
    project = json.load(response)
print('Project:', project.get('name'))
print('Coordinate frame:', project.get('coordinate_frame'))
print('Time scale:', project.get('time_scale'))
for site in project['sites']:
    terrain = site.get('terrain') or {}
    print('\n' + site['name'])
    print('  Terrain:', terrain.get('source', 'NOT ADDED'))
    print('  Synthetic terrain:', terrain.get('synthetic', False))
    print('  Tracks:', site.get('track_source', 'NOT ADDED'))
    print('  Track mode:', site.get('track_mode'))
    print('  Actual geometry source:', site.get('track_metadata', {}).get('actual_source', 'external / teaching'))
    print('  Geometry model:', site.get('track_metadata', {}).get('model', 'not recorded'))
    print('  Horizon directions:', len(terrain.get('azimuth_deg', [])))
    print('  Range samples per direction:', len(terrain.get('distances_m', [])))
print('\nSaved mission settings:', project.get('ui_state', {}))

## 3. Download my workspace — do this before disconnecting

This saves the data and the latest settings saved by the interface. It is also a reliable
alternative if a download from the embedded application is blocked by your browser.

In [ ]:
from datetime import datetime, timezone
with urllib.request.urlopen(BASE + '/api/export', timeout=20) as response:
    project = json.load(response)
stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_UTC')
export_path = Path('/content') / f'LunaShift_workspace_{stamp}.json'
export_path.write_text(json.dumps(project, indent=2), encoding='utf-8')
files.download(str(export_path))
print('Keep this JSON file. Open it in Data & Sources on your next session.')

## 4. Optional: reopen a saved workspace

You can also use **Data & Sources → Open a saved workspace** inside the app.
This cell is a fallback for browsers that restrict file selection inside an iframe.
After loading, run the final line to display a fresh app view with the restored settings.

In [ ]:
from urllib.request import Request
saved = files.upload()
json_names = [name for name in saved if name.lower().endswith('.json')]
if len(json_names) != 1:
    raise RuntimeError('Upload exactly one exported LunaShift project JSON.')
with urllib.request.urlopen(BASE + '/api/state', timeout=20) as response:
    state = json.load(response)
payload = {'kind': 'project', 'project': json.loads(saved[json_names[0]].decode('utf-8'))}
request = Request(BASE + '/api/import', data=json.dumps(payload).encode(),
                  headers={'Content-Type':'application/json','X-LunaShift-Token':state['token']})
try:
    with urllib.request.urlopen(request, timeout=30) as response:
        result = json.load(response)
    print(result['message'])
except Exception as error:
    raise RuntimeError('The project was not loaded. Check that it is an exported LunaShift JSON.') from error
output.serve_kernel_port_as_iframe(PORT, height='1100')

## 5. Optional: run the reproducible Python tests

The basic platform uses only Python's standard library. Terrain converter tests may be skipped
when Rasterio, NumPy, and PyProj are not installed. No need to install them just to learn the app.

In [ ]:
result = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
                        cwd=PROJECT_ROOT, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
if result.returncode:
    raise RuntimeError('A test failed. Share the output with your team.')

## Optional: test source connectivity

Run once to check map services, a short actual Horizons query and the DEM headers on this runtime. It does not download the full terrain file. Failures are reported rather than hidden. The full output is saved to `source_check.json`.


In [ ]:
# Optional: check actual NASA responses without changing the project.
import json, urllib.request
from google.colab import files
report = json.load(urllib.request.urlopen(BASE + "/api/diagnostics", timeout=45))
print(json.dumps(report, indent=2))
path = "/content/LunaShift_connection_report.json"
with open(path, "w") as handle:
    json.dump(report, handle, indent=2)
files.download(path)


## Optional: check the actual SPICE setup
This uses the installed library and downloaded kernels, not test fixtures. It generates tracks for three sites and dates and checks basic numerical validity. It is not a scientific accuracy certificate. Add `--online` to also request a short Horizons comparison. You can do the same comparison in **Data & Sources → SPICE made simple**.


In [ ]:
result = subprocess.run([sys.executable, "tools/verify_spice.py"],
                        cwd=PROJECT_ROOT, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
if result.returncode:
    print("The actual-kernel check did not pass. Read the setup message above; no replacement sky was invented.")


## 6. Diagnostics / display the app again

Keep this log when reporting a problem. A NASA request can fail because of an external
network, time range, certificate, or service response; synthetic data is not used as a fallback.

In [ ]:
print(LOG_PATH.read_text(encoding='utf-8')[-10000:] if LOG_PATH.exists() else 'No server log yet.')
if 'LUNASHIFT_PROCESS' in globals() and LUNASHIFT_PROCESS.poll() is None:
    output.serve_kernel_port_as_iframe(PORT, height='1100')
else:
    print('The process is not running. Re-run the upload-and-launch cell.')

## 7. Stop

Download your workspace first. Autosave in `/content` does not survive deletion of the runtime.

In [ ]:
if 'LUNASHIFT_PROCESS' in globals() and LUNASHIFT_PROCESS.poll() is None:
    LUNASHIFT_PROCESS.terminate()
    try:
        LUNASHIFT_PROCESS.wait(timeout=5)
    except subprocess.TimeoutExpired:
        LUNASHIFT_PROCESS.kill()
        LUNASHIFT_PROCESS.wait(timeout=5)
    LUNASHIFT_LOG.close()
    print('LunaShift stopped.')
else:
    print('No LunaShift process is running.')

### Technical notes

The notebook uses Google's [supported iframe display helper](https://github.com/googlecolab/colabtools/blob/main/google/colab/output/_util.py).
The desktop application remains a local standard-library Python server with HTML/CSS/JavaScript.
This notebook is structurally validated and its Python cells compile. A live, signed-in Google Colab
session has not been executed by the build environment. The application test report documents the
checks that were actually run. Existing schema-v1 project JSON is supported.